### Vector stores and retrievers
This video tutorial will familiarize you with LangChain's vector store and retriever abstractions. These abstractions are designed to support retrieval of data-- from (vector) databases and other sources-- for integration with LLM workflows. They are important for applications that fetch data to be reasoned over as part of model inference, as in the case of retrieval-augmented generation.

We will cover 
- Documents
- Vector stores
- Retrievers


### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:

- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata.
The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.

Let's generate some sample documents:

In [62]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [63]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [64]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

# llm = ChatGroq(groq_api_key=groq_api_key, model="openai/gpt-oss-20b")
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="qwen3:8b"
)
llm

ChatOllama(model='qwen3:8b')

In [65]:
import sys
print(sys.executable)

d:\MindsprintCourse\GenAI_Apps\chatbotWithMessageHistory\venv\python.exe


In [66]:
## convert documents to vector db
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="qwen3-embedding"
)


# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")




In [67]:
## vector store 
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents,
    embedding=embeddings,
    collection_name="my_4096_collection",
    persist_directory="./chroma_4096_db"
)

In [68]:
collection = vectorstore._collection
data = collection.get()

print(data)

{'ids': ['bdd49b1e-d9dd-46b5-9d6f-fd7f098e5f5d', 'db4df4a7-e138-4d59-aeb1-2c675fb0c722', 'dc9900b7-4a27-4554-8333-bf232b4996a8', 'e95f77d0-3261-4326-b677-443b8bf8b7cb', '68cacab1-0071-4e94-a2b6-605f09301634', '72c649fc-6b34-4400-ab62-bfa8bd974733', '86c0caf4-344c-4130-8ac2-dc622de05a40', 'cc6addba-35e0-4031-be60-c72f2631b0b2', 'c0e6ce0e-032b-4045-a207-0bcdd670f0fd', 'd95ee949-1365-4e81-a08a-f643aac8d6c3', 'e588217e-cc46-44b5-9875-cea4429803ed', 'da2d0887-d7c7-43c8-97fc-24900f9aae56', '5cb68e93-0268-4916-9904-15498bef6efd', '9d17736c-5896-4050-aa47-fe89d3505150', '501b3c0f-f9cf-4f3d-85d4-1b0931d31b91', '72db9f31-937f-49ce-a3f0-311fdd0d1aa8', '5205d2d7-d3d5-4aa8-8c22-5d2ef38df6cf', '173b762d-efb8-48b9-a24f-6f57048512b3', '927a79dc-f988-45ea-934d-98b0dbd38776', '956839db-6558-4ed9-b8b1-39e2b3d7502d', 'a0090418-3238-48a4-bd35-c77b1cf96d31', '39450a45-f79b-4a6d-b493-fa70918cb7f0', '684aeb2b-a6da-4815-8a52-b36787a2090c', '6204dd53-5ada-401e-a8b8-d0595e31c746', '93f0060c-366b-4cf7-8800-25aa7d

In [69]:

vectorstore

In [70]:
vectorstore.similarity_search("What are the pets that are social animals?")

[Document(id='72c649fc-6b34-4400-ab62-bfa8bd974733', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='95abb8da-9b86-4f8d-ad34-9a9ce426c91a', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='49bf7073-e3f3-44a3-85c2-547f41efe6e4', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='f258f966-0da7-4ac4-b904-bc4023095232', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [71]:
# async query
await vectorstore.asimilarity_search("What are the pets that are social animals?")

[Document(id='72c649fc-6b34-4400-ab62-bfa8bd974733', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='95abb8da-9b86-4f8d-ad34-9a9ce426c91a', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='49bf7073-e3f3-44a3-85c2-547f41efe6e4', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='f258f966-0da7-4ac4-b904-bc4023095232', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [72]:
vectorstore.similarity_search_with_score("What are the pets that are social animals?")

[(Document(id='72c649fc-6b34-4400-ab62-bfa8bd974733', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  0.8724915981292725),
 (Document(id='95abb8da-9b86-4f8d-ad34-9a9ce426c91a', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  0.8724915981292725),
 (Document(id='49bf7073-e3f3-44a3-85c2-547f41efe6e4', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  0.8724915981292725),
 (Document(id='f258f966-0da7-4ac4-b904-bc4023095232', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  0.8724915981292725)]

### Retrievers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

In [73]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)

retriever.batch(["cat","dog"])

[[Document(id='8ddd5451-ff0e-4c8f-877b-69cc1efadff6', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='95abb8da-9b86-4f8d-ad34-9a9ce426c91a', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [74]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}
)

retriever.batch(["cat","dogs"])

[[Document(id='8ddd5451-ff0e-4c8f-877b-69cc1efadff6', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='95abb8da-9b86-4f8d-ad34-9a9ce426c91a', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [75]:
## RAG
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human", message)])
retriever = vectorstore.as_retriever()

rag_chain={"context":retriever,"question":RunnablePassthrough()}|prompt|llm

response=rag_chain.invoke("tell me about dogs")
print(response.content)


Based on the provided context, dogs are described as great companions known for their loyalty and friendliness. The information is consistently repeated across multiple documents from the "mammal-pets-doc" source, emphasizing these traits without elaborating on other characteristics or details.


Based on the provided context, dogs are described as great companions known for their loyalty and friendliness. The information is consistently repeated across multiple documents from the "mammal-pets-doc" source, emphasizing these traits without elaborating on other characteristics or details.




